<div style="border-left:4px solid #2980b9;padding-left:16px;margin-bottom:22px;">
  <p style="margin:0;color:#64748b;font-size:13px;letter-spacing:.7px;"><b>GREENFORGE AI · ML / DECISION SUPPORT / 03</b></p>
  <h1 style="margin:9px 0 6px;color:#1e3a8a;">☀️ When should the machine run?</h1>
  <p style="font-size:17px;color:#334155;margin:0;">Smart, solar-aware machine scheduling with mixed-integer linear programming (MILP)</p>
</div>

**Our question:** If one machine job can move to another time, **when would it need the least expensive electricity from the grid while making good use of solar power?** And what should Notebook 02 tell us before recommending a run?

**At a glance:** We compare the *earliest feasible start* with a *model-selected start*. Every option performs the **same job**, uses the **same amount of machine energy**, and respects stated time and grid limits. The figures are **illustrative historical what-if estimates**, not factory instructions or proven cost/CO₂ savings.

**What makes this honest:** Notebook 01 contains rolling, **one-step (15-minute) historical test predictions**; it does **not** produce a multi-hour forecast known at one decision time. Notebook 02 produces **retrospective observations** that become known only at `available_at`. Where they cannot be joined safely, this notebook says so instead of creating false forecast coverage.

**Journey:** data handoff → assumptions → feasible hours → MILP → cost/solar trade-off → safety and uncertainty → plain-English answers.

### 🛠️ 1. Setup — what do we need?

Only standard scientific Python libraries are used: `numpy`, `pandas`, `scipy`, `plotly`, and a Parquet reader such as `pyarrow` (already needed by the earlier notebooks). Run this notebook from **inside the same GreenForge AI project** after Notebooks 01 and 02. No ETL files are changed.

In [1]:
import os
os.environ.setdefault("POLARS_MAX_THREADS", "4")

from pathlib import Path
from html import escape
import json
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML
from scipy.optimize import Bounds, LinearConstraint, milp

warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "etl_config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the GreenForge AI project (etl_config.json is required).")
OUT = ROOT / "reports" / "ml_short_term" / "notebook_runs"
OUT.mkdir(parents=True, exist_ok=True)
BLUE, ORANGE, GREEN, PURPLE = "#2563eb", "#f59e0b", "#16a34a", "#7c3aed"
RED, SLATE, MUTED = "#dc2626", "#334155", "#64748b"
SLOT_MINUTES = 15
SLOT_HOURS = SLOT_MINUTES / 60
TIMEZONE_NOTE = "Europe/Berlin publisher wall-clock (naive timestamps)"

def report(text):
    display(Markdown("> **💡 What this tells us**\n>\n" +
                     "\n".join("> " + line for line in str(text).strip().splitlines())))

def show_cards(cards):
    blocks = []
    for title, value, note in cards:
        blocks.append(
            "<div style='background:#f8fafc;border:1px solid #e2e8f0;"
            "border-radius:14px;padding:18px 19px;min-width:0;'>"
            f"<div style='font-size:13px;color:#64748b;margin-bottom:8px'>{escape(str(title))}</div>"
            f"<div style='font-size:24px;font-weight:700;color:#334155;margin-bottom:7px;overflow-wrap:anywhere'>{escape(str(value))}</div>"
            f"<div style='font-size:12px;color:#475569;line-height:1.45'>{escape(str(note))}</div></div>"
        )
    display(HTML("<div style='display:grid;grid-template-columns:repeat(3,minmax(190px,1fr));"
                 "gap:13px;margin:13px 0 22px'>" + "".join(blocks) + "</div>"))

def show_table(frame, columns=None, caption=None, max_rows=20):
    if caption:
        display(Markdown(f"**{caption}**"))
    table = frame.copy() if columns is None else frame.loc[:, columns].copy()
    if table.empty:
        display(Markdown("*No matching rows. This is not evidence that the equipment is healthy.*"))
        return
    for column in table.select_dtypes(include="datetime").columns:
        table[column] = table[column].dt.strftime("%d %b %Y %H:%M")
    fmt = {}
    for column in table.select_dtypes(include="number").columns:
        if any(word in column.lower() for word in ("cost", "eur", "kg", "kwh", "kw", "price", "score", "share", "pct", "fraction")):
            fmt[column] = "{:.4f}"
        else:
            fmt[column] = "{:,.0f}"
    styled = (table.head(max_rows).style.format(fmt, na_rep="—")
              .set_table_styles([
                  {"selector":"th", "props":[("background-color","#eff6ff"),("color","#1e3a8a"),("padding","8px 10px"),("font-weight","600")]},
                  {"selector":"td", "props":[("padding","7px 10px"),("border-bottom","1px solid #eef2f7")]},
                  {"selector":"table", "props":[("font-family","Arial,sans-serif"),("font-size","13px"),("border-collapse","collapse")]}
              ]).hide(axis="index"))
    display(styled)

def polish(fig, title, subtitle="", height=430):
    fig.update_layout(title={"text": f"{title}<br><sup>{subtitle}</sup>", "x":0.045},
        height=height, template="plotly_white", plot_bgcolor="white", paper_bgcolor="white",
        font={"family":"Arial,sans-serif","color":SLATE,"size":12},
        margin={"l":65,"r":55,"t":90,"b":55},
        legend={"orientation":"h","yanchor":"bottom","y":1.03,"xanchor":"right","x":1.0},
        hovermode="x unified")
    fig.update_xaxes(showgrid=True, gridcolor="#e2e8f0", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor="#e2e8f0", zeroline=False)
    fig.show()

def local_clock(values):
    """Do not silently UTC-localize already naive publisher-wall-clock dates."""
    parsed = pd.to_datetime(values, errors="coerce")
    if getattr(parsed.dt, "tz", None) is not None:
        parsed = parsed.dt.tz_convert("Europe/Berlin").dt.tz_localize(None)
    return parsed

show_cards([
    ("Decision type", "What-if MILP", "One flexible, non-interruptible machine job"),
    ("Time resolution", "15 min", "All power readings refer to interval START"),
    ("Key inputs", "01 + 02 + SMARD", "PV forecast, machine context, anomaly advisory and prices"),
])

### ❓ 2. Can we connect Notebook 01 and Notebook 02 without inventing forecasts?

Notebook 01 saves `machine_predictions_15min.csv` and `pv_predictions_15min.csv`. I read from the configured `reports/ml_short_term/notebook_runs`. Each predicted value belongs to `target_time` and was made at the **previous** interval. Notebook 02 saves `anomaly_review_15min.parquet`; a warning can only be consulted from its `available_at` time onward.

We check their dates and explain how much real overlap exists. **A prediction made at 10:15 for 10:15–10:30 is not a forecast that could have been known at 08:00.**

In [2]:
FORECAST_PATHS = {"machine": OUT / "machine_predictions_15min.csv",
                  "pv": OUT / "pv_predictions_15min.csv"}
ADVISORY_PATH = OUT / "anomaly_review_15min.parquet"
PRICE_COLUMN = "DayAhead_Price_EUR_MWh"

def load_forecast(kind):
    path = FORECAST_PATHS[kind]
    if not path.is_file():
        return pd.DataFrame(columns=["target_time", f"{kind}_forecast_kw"]), f"Missing: {path.name}"
    df = pd.read_csv(path)
    needed = {"WsDateTime", "target_time", "prediction"}
    if not needed.issubset(df.columns):
        raise ValueError(f"{path.name}: expected {sorted(needed)}")
    df["WsDateTime"] = local_clock(df["WsDateTime"])
    df["target_time"] = local_clock(df["target_time"])
    if df[["WsDateTime", "target_time"]].isna().any().any():
        raise ValueError(f"{path.name}: invalid timestamp")
    if not (df["target_time"] - df["WsDateTime"] == pd.Timedelta(minutes=15)).all():
        raise ValueError(f"{path.name}: prediction interval is not 15 minutes ahead")
    if df.target_time.duplicated().any():
        raise ValueError(f"{path.name}: repeated target timestamps")
    df["prediction"] = pd.to_numeric(df["prediction"], errors="coerce")
    df["prediction"] = df["prediction"].replace([np.inf, -np.inf], np.nan)
    negatives = int(df.prediction.lt(0).sum())
    # A power forecast cannot yield negative available generation. Only PV is clipped.
    if kind == "pv":
        df["prediction"] = df["prediction"].clip(lower=0)
    cleaned = df[["target_time", "prediction"]].rename(
        columns={"prediction": f"{kind}_forecast_kw"})
    return cleaned.sort_values("target_time").reset_index(drop=True), (
        f"{len(cleaned):,} historical 15-minute predictions"
        + (f" · {negatives} negative PV estimates set to 0" if negatives and kind == "pv" else ""))

forecasts = {}
forecast_notes = {}
for key in ("machine", "pv"):
    forecasts[key], forecast_notes[key] = load_forecast(key)

if ADVISORY_PATH.is_file():
    try:
        advisory = pd.read_parquet(ADVISORY_PATH)
    except ImportError as exc:
        raise ImportError("Install pyarrow in your project environment to read the Notebook 02 Parquet output") from exc
    required = {"system", "WsDateTime", "available_at", "advisory", "review_flag"}
    if not required.issubset(advisory.columns):
        raise ValueError(f"Notebook 02 advisory missing columns: {sorted(required-set(advisory.columns))}")
    for col in ("WsDateTime", "available_at"):
        advisory[col] = local_clock(advisory[col])
    advisory = advisory.sort_values(["system", "available_at"]).reset_index(drop=True)
    if (advisory.available_at < advisory.WsDateTime + pd.Timedelta(minutes=15)).any():
        raise ValueError("Notebook 02 contains an advisory available before the measurement ended")
    advisory_note = f"{len(advisory):,} timestamped historical condition records"
else:
    advisory = pd.DataFrame(columns=["system", "WsDateTime", "available_at", "advisory", "review_flag"])
    advisory_note = "Missing — run Notebook 02 first; no condition clearance is assumed"

if not forecasts["machine"].empty and not forecasts["pv"].empty:
    paired = forecasts["machine"].merge(forecasts["pv"], on="target_time", how="inner")
    paired_count = int(paired[["machine_forecast_kw", "pv_forecast_kw"]].notna().all(axis=1).sum())
else:
    paired_count = 0

handoff_table = pd.DataFrame([
    {"Input": "01 · TEC 15-minute predictions", "File": FORECAST_PATHS["machine"].name,
     "Rows": len(forecasts["machine"]), "Status": forecast_notes["machine"]},
    {"Input": "01 · PV 15-minute predictions", "File": FORECAST_PATHS["pv"].name,
     "Rows": len(forecasts["pv"]), "Status": forecast_notes["pv"]},
    {"Input": "02 · Timestamped advisory", "File": ADVISORY_PATH.name,
     "Rows": len(advisory), "Status": advisory_note},
])
show_table(handoff_table, caption="Actual outputs from the previous notebooks")
show_cards([
    ("Matching 01 forecast intervals", f"{paired_count:,}", "TEC and PV historical test predictions at identical times"),
    ("Anomaly handoff", "Available" if not advisory.empty else "Missing", "Read only after available_at"),
    ("Planning horizon", "Not forecasted", "One-step predictions are not a full multi-hour forecast"),
])
report("Notebook 01 forecasts are historical **rolling one-step predictions**, and can be inspected together "
       "only when their target times match. If the two held-out periods do not overlap, we do not pretend "
       "there is a simultaneous TEC + PV forecast. Notebook 02's review scores are observations, not "
       "predictions of future equipment failure.")

**Actual outputs from the previous notebooks**

Input,File,Rows,Status
01 · TEC 15-minute predictions,machine_predictions_15min.csv,"6,584","6,584 historical 15-minute predictions"
01 · PV 15-minute predictions,pv_predictions_15min.csv,"1,651","1,651 historical 15-minute predictions · 426 negative PV estimates set to 0"
02 · Timestamped advisory,anomaly_review_15min.parquet,"70,272","70,272 timestamped historical condition records"


> **💡 What this tells us**
>
> Notebook 01 forecasts are historical **rolling one-step predictions**, and can be inspected together only when their target times match. If the two held-out periods do not overlap, we do not pretend there is a simultaneous TEC + PV forecast. Notebook 02's review scores are observations, not predictions of future equipment failure.

### ❓ 3. What rules does a realistic scheduling question need?

Here is the **clearly labeled what-if contract**. Assume a single job needs a fixed amount of power for one uninterrupted hour. It can start within a daytime window. We do **not** add the TEC forecast to the job demand: that would risk counting the same machine twice. We keep the TEC forecast for context and model-readiness checks; **job consumption needs verified production records**.

**Important:** The publisher data do not verify that TEC_48S and IPE_PV feed the same behind-the-meter electrical bus. The default scenario intentionally *assumes* they do, and later compares a **no-sharing** case. Neither is an established plant configuration. The default carbon weight is a transparent preference, not an electricity tariff or a carbon market price.

In [3]:
# EDIT THIS ONE BLOCK to test a different hypothetical production contract.
SCENARIO = {
    "job_kw": 1.20,                  # Illustrative fixed job load; NOT a forecast from Notebook 01
    "job_slots": 4,                 # Four consecutive 15-minute intervals = one hour
    "window_start_hour": 8,
    "window_end_hour": 18,         # Last slot runs 17:45–18:00
    "grid_cap_kw": 4.0,            # Hypothetical feeder headroom (kW), requires verification
    "other_load_kw": 0.0,          # Separate site baseload; NOT the TEC machine forecast
    "assume_shared_pv_bus": True,  # Unverified: what-if electrical topology
    "carbon_shadow_eur_per_kg": 0.12,   # Hypothetical preference, NOT a carbon levy
    "annual_grid_factor_kg_per_kwh": 0.363,  # Illustrative STATIC factor; no hourly carbon claim
    "allowed_weekdays": (0, 1, 2, 3, 4), # Assumed Mon–Fri work days (NOT recorded shifts)
    "blocked_start_times": (),   # e.g. ("12:00", "12:15") -- manually approved exclusions
    "operator_approved": False,  # Never infer approval from no anomaly flag
    "historical_date": None,     # e.g. "2024-04-03"; None picks an informative covered day
    "prefer_pv_forecast": True,
    "allow_illustrative_tariff_if_missing": True,
}
assert SCENARIO["job_kw"] > 0 and 1 <= SCENARIO["job_slots"] <= 16
assert SCENARIO["window_start_hour"] < SCENARIO["window_end_hour"] <= 24
assert SCENARIO["other_load_kw"] >= 0 and SCENARIO["grid_cap_kw"] > 0

assumptions = pd.DataFrame([
    {"Rule": "Flexible job", "What it means": f"{SCENARIO['job_kw']:.2f} kW, {SCENARIO['job_slots']*15} consecutive minutes"},
    {"Rule": "Search window", "What it means": f"{SCENARIO['window_start_hour']:02}:00–{SCENARIO['window_end_hour']:02}:00, assumed weekdays"},
    {"Rule": "Grid capacity", "What it means": f"{SCENARIO['grid_cap_kw']:.2f} kW hypothetical headroom"},
    {"Rule": "PV electrical sharing", "What it means": "Assumed yes for the main what-if; compared with no-sharing"},
    {"Rule": "Decision priority", "What it means": "Wholesale job cost + stated carbon shadow of grid imports"},
    {"Rule": "Maintenance and safety", "What it means": "Notebook 02 advisory requires review; it is NOT an automatic stop command"},
    {"Rule": "Operator approval", "What it means": "Not supplied — outputs are only candidate timing ideas"},
])
show_table(assumptions, caption="Transparent job, grid and safety assumptions")
report("The optimization **does not make a production schedule from missing factory records**. "
       "All hours, job size, electrical sharing, headroom and carbon preferences are explicit what-if assumptions. "
       "They must be replaced with approved operating inputs before live use.")

**Transparent job, grid and safety assumptions**

Rule,What it means
Flexible job,"1.20 kW, 60 consecutive minutes"
Search window,"08:00–18:00, assumed weekdays"
Grid capacity,4.00 kW hypothetical headroom
PV electrical sharing,Assumed yes for the main what-if; compared with no-sharing
Decision priority,Wholesale job cost + stated carbon shadow of grid imports
Maintenance and safety,Notebook 02 advisory requires review; it is NOT an automatic stop command
Operator approval,Not supplied — outputs are only candidate timing ideas


> **💡 What this tells us**
>
> The optimization **does not make a production schedule from missing factory records**. All hours, job size, electrical sharing, headroom and carbon preferences are explicit what-if assumptions. They must be replaced with approved operating inputs before live use.

### ❓ 4. Which historical day has enough real solar and price information?

We use the archived 15-minute PV feature table for actual measured generation and, **when present**, the ETL's SMARD day-ahead price column. Where Notebook 01 has a PV test prediction for the same historical slot, we use that **rolling historical forecast** for the exploratory scenario. The rest of the slots may have been predicted at different origins: **this is a retrospective opportunity study**, *not* an executable schedule created at a single earlier instant.

**2024 market detail:** German day-ahead wholesale prices were **hourly** in 2024, even when our decision slots are 15 minutes. Quarter-hour day-ahead products began in October 2025. Repeated hourly prices across four slots therefore do **not** mean we have four independent price observations.

In [4]:

# ================================================================
# SECTION 4 — SELECT A COMPLETE HISTORICAL 15-MINUTE WINDOW
# Fix: robust timestamp comparison across datetime precisions.
# Never fill missing timestamps, PV readings, or electricity prices.
# ================================================================

def read_eda(system):
    path = (
        ROOT / "data" / "processed" / "features"
        / system / "eda_15min.parquet"
    )

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run the ETL pipeline first."
        )

    try:
        import pyarrow.parquet as pq
    except ImportError as exc:
        raise ImportError(
            "Install pyarrow in the GreenForge AI environment."
        ) from exc

    available = set(pq.read_schema(path).names)
    required = {"WsDateTime", "power_kw"}

    if not required.issubset(available):
        raise ValueError(
            f"{system}: missing {sorted(required - available)}"
        )

    columns = ["WsDateTime", "power_kw"]

    if PRICE_COLUMN in available:
        columns.append(PRICE_COLUMN)

    frame = pd.read_parquet(path, columns=columns)

    frame["WsDateTime"] = local_clock(frame["WsDateTime"])

    if frame["WsDateTime"].isna().any():
        raise ValueError(
            f"{system}: invalid timestamps in feature data."
        )

    if frame["WsDateTime"].duplicated().any():
        raise ValueError(
            f"{system}: duplicate 15-minute timestamps."
        )

    frame["power_kw"] = pd.to_numeric(
        frame["power_kw"], errors="coerce"
    )

    if PRICE_COLUMN not in frame.columns:
        frame[PRICE_COLUMN] = np.nan

    frame[PRICE_COLUMN] = pd.to_numeric(
        frame[PRICE_COLUMN], errors="coerce"
    )

    frame[["power_kw", PRICE_COLUMN]] = (
        frame[["power_kw", PRICE_COLUMN]]
        .replace([np.inf, -np.inf], np.nan)
    )

    return (
        frame.sort_values("WsDateTime")
        .reset_index(drop=True)
    )


# ------------------------------------------------
# 1. Read historical machine and solar information
# ------------------------------------------------

pv_raw = read_eda("IPE_PV")
tec_raw = read_eda("TEC_48S")

pv_raw = pv_raw.rename(columns={
    "power_kw": "pv_measured_kw",
    PRICE_COLUMN: "price_pv_eur_mwh",
})

tec_raw = tec_raw.rename(columns={
    "power_kw": "tec_measured_kw",
    PRICE_COLUMN: "price_tec_eur_mwh",
})


# ------------------------------------------------
# 2. Combine history with Notebook 01 predictions
# ------------------------------------------------

all_slots = pv_raw.merge(
    tec_raw,
    on="WsDateTime",
    how="outer",
    validate="one_to_one",
)

for system in ("pv", "machine"):
    forecast_frame = forecasts[system].rename(
        columns={"target_time": "WsDateTime"}
    )

    all_slots = all_slots.merge(
        forecast_frame,
        on="WsDateTime",
        how="left",
        validate="one_to_one",
    )

all_slots = (
    all_slots.sort_values("WsDateTime")
    .reset_index(drop=True)
)


# ------------------------------------------------
# 3. Check electricity-price consistency
# ------------------------------------------------

both_prices = (
    all_slots[
        ["price_pv_eur_mwh", "price_tec_eur_mwh"]
    ]
    .notna()
    .all(axis=1)
)

conflicts = (
    both_prices
    & (
        all_slots["price_pv_eur_mwh"]
        - all_slots["price_tec_eur_mwh"]
    ).abs().gt(0.0001)
)

all_slots["price_eur_mwh"] = (
    all_slots["price_pv_eur_mwh"]
    .combine_first(all_slots["price_tec_eur_mwh"])
)

# Conflicting real prices must not be used.
all_slots.loc[
    conflicts, "price_eur_mwh"
] = np.nan

all_slots["price_source"] = np.where(
    conflicts,
    "conflict/unavailable",
    np.where(
        all_slots["price_eur_mwh"].notna(),
        "ETL SMARD archive",
        "unavailable",
    ),
)

# Negative measured PV cannot represent usable generation.
all_slots.loc[
    all_slots["pv_measured_kw"] < 0,
    "pv_measured_kw",
] = np.nan


# ------------------------------------------------
# 4. Restrict data to eligible days and hours
# ------------------------------------------------

all_slots["date"] = (
    all_slots["WsDateTime"].dt.normalize()
)

all_slots["hour"] = (
    all_slots["WsDateTime"].dt.hour
)

window_start = SCENARIO["window_start_hour"]
window_end = SCENARIO["window_end_hour"]

slots_per_window = (window_end - window_start) * 4

window_ok = all_slots["hour"].between(
    window_start, window_end - 1
)

workday_ok = (
    all_slots["WsDateTime"]
    .dt.dayofweek
    .isin(SCENARIO["allowed_weekdays"])
)

possible = all_slots.loc[
    window_ok & workday_ok
].copy()


# ------------------------------------------------
# 5. FIX: Check the actual 15-minute grid
# ------------------------------------------------

def exact_15min_grid(times, day, expected_slots):
    """
    Check exact timestamps without relying on Series.equals().

    Datetime64[us] and datetime64[ns] can represent identical
    timestamps but fail a strict pandas dtype comparison.

    Both sides are compared at nanosecond precision.

    This function does NOT create or fill missing timestamps.
    """

    actual = pd.to_datetime(
        times, errors="coerce"
    )

    if pd.isna(actual).any():
        return False

    if len(actual) != expected_slots:
        return False

    actual_grid = pd.DatetimeIndex(
        actual
    ).sort_values()

    expected_grid = pd.date_range(
        start=(
            pd.Timestamp(day).normalize()
            + pd.Timedelta(hours=window_start)
        ),
        periods=expected_slots,
        freq="15min",
    )

    actual_values = actual_grid.to_numpy(
        dtype="datetime64[ns]"
    )

    expected_values = expected_grid.to_numpy(
        dtype="datetime64[ns]"
    )

    return bool(
        np.array_equal(
            actual_values,
            expected_values,
        )
    )


# ------------------------------------------------
# 6. Evaluate data coverage for every possible day
# ------------------------------------------------

possible["pv_forecast_usable"] = (
    np.isfinite(possible["pv_forecast_kw"])
    & possible["pv_forecast_kw"].ge(0)
)

possible["pv_measured_usable"] = (
    np.isfinite(possible["pv_measured_kw"])
)

possible["real_price_usable"] = (
    np.isfinite(possible["price_eur_mwh"])
)

coverage = (
    possible.groupby("date")
    .agg(
        slots=("WsDateTime", "size"),
        forecast_slots=("pv_forecast_usable", "sum"),
        measured_slots=("pv_measured_usable", "sum"),
        priced_slots=("real_price_usable", "sum"),
        solar_kw_sum=(
            "pv_measured_kw",
            lambda s: float(
                s.clip(lower=0).sum(min_count=1)
            ),
        ),
    )
    .reset_index()
)

# Check each date by its exact timestamp sequence.
grid_checks = {
    pd.Timestamp(day): exact_15min_grid(
        group["WsDateTime"],
        day,
        slots_per_window,
    )
    for day, group in possible.groupby("date")
}

coverage["grid_ok"] = (
    coverage["date"]
    .map(grid_checks)
    .fillna(False)
    .astype(bool)
)

total_days_checked = len(coverage)

coverage = coverage.loc[
    (coverage["slots"] == slots_per_window)
    & coverage["grid_ok"]
].copy()

print(
    f"15-minute grid validation: "
    f"{len(coverage)} complete days "
    f"out of {total_days_checked} eligible days."
)

if coverage.empty:
    raise ValueError(
        "No complete historical 15-minute window was found. "
        "Check the selected hours and feature timestamp alignment. "
        "Missing intervals will not be filled automatically."
    )


# ------------------------------------------------
# 7. Select a suitable historical scenario date
# ------------------------------------------------

if SCENARIO["historical_date"] is not None:

    selected_date = pd.Timestamp(
        SCENARIO["historical_date"]
    ).normalize()

    chosen_days = coverage.loc[
        coverage["date"] == selected_date
    ]

    if chosen_days.empty:
        raise ValueError(
            f"{selected_date:%Y-%m-%d} does not have a "
            "complete 15-minute daytime grid. "
            "Choose another historical_date or use None."
        )

    day_rule = "user-selected complete historical date"

else:

    forecast_full = coverage.loc[
        coverage["forecast_slots"] == slots_per_window
    ]

    measured_full = coverage.loc[
        coverage["measured_slots"] == slots_per_window
    ]

    if (
        SCENARIO["prefer_pv_forecast"]
        and not forecast_full.empty
    ):
        eligible_days = forecast_full.copy()
        day_rule = (
            "complete historical PV test predictions"
        )

    elif not measured_full.empty:
        eligible_days = measured_full.copy()
        day_rule = (
            "complete historical PV measurements"
        )

    else:
        raise ValueError(
            "No complete daytime PV data are available. "
            "Missing PV must not be replaced with zero."
        )

    # Prioritize available real prices, then a sunny example.
    eligible_days["solar_kw_sum"] = (
        eligible_days["solar_kw_sum"].fillna(-1)
    )

    eligible_days = eligible_days.sort_values(
        ["priced_slots", "solar_kw_sum", "date"],
        ascending=[False, False, True],
    )

    selected_date = pd.Timestamp(
        eligible_days.iloc[0]["date"]
    )


# ------------------------------------------------
# 8. Build the final historical scenario window
# ------------------------------------------------

window = (
    possible.loc[
        possible["date"] == selected_date
    ]
    .sort_values("WsDateTime")
    .reset_index(drop=True)
    .copy()
)

# Final independent grid check.
if not exact_15min_grid(
    window["WsDateTime"],
    selected_date,
    slots_per_window,
):
    raise ValueError(
        "The selected day has genuinely missing, duplicate, "
        "or misaligned 15-minute timestamps. "
        "No schedule will be calculated on a broken grid."
    )


# ------------------------------------------------
# 9. Choose one consistent PV input source
# ------------------------------------------------

if (
    SCENARIO["prefer_pv_forecast"]
    and window["pv_forecast_kw"].notna().all()
    and window["pv_forecast_kw"].ge(0).all()
):

    window["pv_input_kw"] = (
        window["pv_forecast_kw"]
    )

    pv_mode = (
        "historical rolling PV test predictions "
        "(different forecast origins)"
    )

elif (
    window["pv_measured_kw"].notna().all()
    and window["pv_measured_kw"].ge(0).all()
):

    window["pv_input_kw"] = (
        window["pv_measured_kw"]
    )

    pv_mode = "historical observed PV (hindsight)"

else:

    raise ValueError(
        "The selected day does not have a complete "
        "PV forecast or measured PV series. "
        "Missing values will not be replaced with zero."
    )


# ------------------------------------------------
# 10. Use actual electricity prices if available
# ------------------------------------------------

if window["price_eur_mwh"].notna().all():

    price_mode = (
        "actual archived SMARD day-ahead wholesale prices"
    )

elif SCENARIO["allow_illustrative_tariff_if_missing"]:

    # Example prices only, NOT measured SMARD values.
    # Identical price for each quarter-hour of an hour.
    illustrative = {
        8: 115,
        9: 100,
        10: 85,
        11: 70,
        12: 45,
        13: 40,
        14: 55,
        15: 80,
        16: 105,
        17: 125,
    }

    missing_hours = sorted(
        set(window["WsDateTime"].dt.hour)
        - set(illustrative)
    )

    if missing_hours:
        raise ValueError(
            "Illustrative prices are not defined for hours "
            f"{missing_hours}. Provide real prices or "
            "extend the explicitly illustrative schedule."
        )

    window["price_eur_mwh"] = (
        window["WsDateTime"]
        .dt.hour
        .map(illustrative)
        .astype(float)
    )

    price_mode = (
        "ILLUSTRATIVE assumed hourly wholesale-like tariff "
        "— NOT SMARD"
    )

else:

    raise ValueError(
        "No complete real SMARD price window exists. "
        "Provide available market prices or explicitly "
        "enable illustrative prices."
    )


# ------------------------------------------------
# 11. Final physical and quality checks
# ------------------------------------------------

if not window[
    ["pv_input_kw", "price_eur_mwh"]
].notna().all().all():
    raise ValueError(
        "PV or electricity-price inputs contain missing values."
    )

if not np.isfinite(
    window[
        ["pv_input_kw", "price_eur_mwh"]
    ].to_numpy(dtype=float)
).all():
    raise ValueError(
        "PV or electricity-price inputs contain infinite values."
    )

if not window["pv_input_kw"].ge(0).all():
    raise ValueError(
        "PV generation cannot be negative."
    )

window["pv_input_source"] = pv_mode
window["price_input_source"] = price_mode


# ------------------------------------------------
# 12. Display the historical scenario summary
# ------------------------------------------------

show_cards([
    (
        "Historical scenario date",
        selected_date.strftime("%d %b %Y"),
        f"Selection rule: {day_rule}",
    ),
    (
        "PV input",
        (
            "ML replay"
            if "predictions" in pv_mode
            else "Observed"
        ),
        pv_mode,
    ),
    (
        "Electricity prices",
        (
            "SMARD archive"
            if price_mode.startswith("actual")
            else "Illustrative"
        ),
        price_mode,
    ),
    (
        "One-date ML overlap",
        (
            f"{int(window['machine_forecast_kw'].notna().sum())}"
            f"/{len(window)}"
        ),
        "TEC historical forecasts present in this window",
    ),
    (
        "Conflicting ETL prices",
        f"{int(conflicts.sum()):,}",
        "Conflicting source prices are excluded",
    ),
    (
        "Daytime grid",
        f"{len(window)} intervals",
        "Validated complete 15-minute grid",
    ),
])

show_table(
    window[
        [
            "WsDateTime",
            "pv_input_kw",
            "price_eur_mwh",
            "machine_forecast_kw",
        ]
    ].head(8),
    caption="First eight intervals of the selected scenario",
)

report(
    f"The selected day is **{selected_date:%d %b %Y}**. "
    f"The PV information comes from **{pv_mode}**, "
    f"and electricity prices come from **{price_mode}**.\n\n"
    "The time grid was checked against every expected "
    "15-minute timestamp, without adding missing intervals.\n\n"
    "This is a **historical scheduling comparison**, "
    "not a live production instruction. "
    "Historical rolling PV predictions come from different "
    "forecast times and cannot be treated as a complete "
    "day-ahead forecast available at 08:00.\n\n"
    "The TEC forecast is shown for context only; "
    "it is not treated as a verified flexible-job load."
)


15-minute grid validation: 262 complete days out of 262 eligible days.


**First eight intervals of the selected scenario**

WsDateTime,pv_input_kw,price_eur_mwh,machine_forecast_kw
25 Mar 2024 08:00,0.9072,99.6800,—
25 Mar 2024 08:15,1.1465,99.6800,—
25 Mar 2024 08:30,2.7588,99.6800,—
25 Mar 2024 08:45,3.0119,99.6800,—
25 Mar 2024 09:00,1.2636,77.4300,—
25 Mar 2024 09:15,2.9607,77.4300,—
25 Mar 2024 09:30,2.6835,77.4300,—
25 Mar 2024 09:45,1.0898,77.4300,—


> **💡 What this tells us**
>
> The selected day is **25 Mar 2024**. The PV information comes from **historical rolling PV test predictions (different forecast origins)**, and electricity prices come from **actual archived SMARD day-ahead wholesale prices**.
> 
> The time grid was checked against every expected 15-minute timestamp, without adding missing intervals.
> 
> This is a **historical scheduling comparison**, not a live production instruction. Historical rolling PV predictions come from different forecast times and cannot be treated as a complete day-ahead forecast available at 08:00.
> 
> The TEC forecast is shown for context only; it is not treated as a verified flexible-job load.

### ❓ 5. What would solar production and grid prices look like together?

Solar power (kW) and electricity price (€/MWh) have **different units**. We plot them on separate axes. A higher solar bar does not guarantee the cheapest hour: the optimizer must consider **how much of the job is actually covered by PV**, the remaining grid import, and the price of that remaining electricity.

In [5]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=window.WsDateTime, y=window.pv_input_kw,
    mode="lines", name="PV available to hypothetical job", line=dict(color=GREEN, width=3)), secondary_y=False)
fig.add_trace(go.Scatter(x=window.WsDateTime, y=window.price_eur_mwh,
    mode="lines", name="Wholesale / assumed price", line=dict(color=BLUE, width=2.5, dash="dot", shape="hv")), secondary_y=True)
fig.update_yaxes(title_text="PV AC power (kW)", secondary_y=False, rangemode="tozero")
fig.update_yaxes(title_text="Electricity price (€/MWh)", secondary_y=True, showgrid=False)
fig.update_xaxes(title="Historical 15-minute interval start · Berlin wall clock")
polish(fig, "When do solar power and prices line up?",
       f"{selected_date:%d %b %Y} · {pv_mode} · {price_mode}", height=450)
report("For the assumed shared electrical connection, PV can cover part of the added job load. "
       "Only the remaining **grid energy** is multiplied by wholesale price. "
       "If TEC and PV are not connected behind the same meter, this solar offset cannot be claimed.")

> **💡 What this tells us**
>
> For the assumed shared electrical connection, PV can cover part of the added job load. Only the remaining **grid energy** is multiplied by wholesale price. If TEC and PV are not connected behind the same meter, this solar offset cannot be claimed.

### ❓ 6. Can Notebook 02 tell us whether this is a good time to run?

**Yes—as a review signal, not a safety certificate.** We look at the latest TEC and PV records available **before the common scheduling decision** (the beginning of the proposed work window). We do **not** use later anomaly scores to rule out earlier starts, because that would reveal future measurements. A lack of an alert is **not** permission to run, especially when a regime was not assessed.

In [6]:
decision_time = window.WsDateTime.iloc[0]

def latest_asof_advisory(records, system, at):
    eligible = records.loc[(records.system == system) & (records.available_at <= at)].copy()
    if eligible.empty:
        return {"System":system, "Latest_available":pd.NaT, "Advisory":"not_available",
                "Age_minutes":np.nan, "Interpretation":"No verified status at decision time"}
    last = eligible.sort_values("available_at").iloc[-1]
    age_min = (pd.Timestamp(at) - pd.Timestamp(last.available_at)).total_seconds()/60
    status = str(last.advisory)
    if status == "review_observation":
        interpretation = "Engineer should inspect a historical anomaly before proposing this job"
    elif status in ("not_assessed", "measurement_unavailable"):
        interpretation = "No reliable anomaly clearance; check measurements and operating status"
    else:
        interpretation = "No model alert in the latest known reading; NOT proof of safety"
    if age_min > 60:
        interpretation = "Latest condition evidence is older than 60 minutes; status is stale"
    return {"System":system, "Latest_available":last.available_at,
            "Advisory":status, "Age_minutes":float(age_min), "Interpretation":interpretation}

condition_rows = [latest_asof_advisory(advisory, k, decision_time) for k in ("TEC_48S", "IPE_PV")]
condition_table = pd.DataFrame(condition_rows)
show_table(condition_table, caption=f"Condition signals known by {decision_time:%d %b %Y %H:%M}")
review_needed = (not SCENARIO["operator_approved"] or
                 any(row["Advisory"] != "no_model_alert" or
                     not np.isfinite(row["Age_minutes"]) or row["Age_minutes"] > 60
                     for row in condition_rows))
show_cards([
    ("Human clearance", "Required" if review_needed else "Provided", "No automatic start from anomaly scores"),
    ("TEC condition", condition_rows[0]["Advisory"], "Latest evidence known at common decision time"),
    ("PV confidence", condition_rows[1]["Advisory"], "Data quality / unusual generation are advisories"),
])
report("Only entries with `available_at <= decision_time` were used. Future observation flags "
       "and forecast errors are excluded. Even with no model alert, job dispatch needs an approved "
       "maintenance, safety, and production check. A review signal does not automatically mean shutdown.")

**Condition signals known by 25 Mar 2024 08:00**

System,Latest_available,Advisory,Age_minutes,Interpretation
TEC_48S,25 Mar 2024 08:00,not_assessed,0,No reliable anomaly clearance; check measurements and operating status
IPE_PV,25 Mar 2024 08:00,review_observation,0,Engineer should inspect a historical anomaly before proposing this job


> **💡 What this tells us**
>
> Only entries with `available_at <= decision_time` were used. Future observation flags and forecast errors are excluded. Even with no model alert, job dispatch needs an approved maintenance, safety, and production check. A review signal does not automatically mean shutdown.

### ❓ 7. How does the MILP choose a feasible machine start?

The solver considers binary decisions: **start at 08:00, 08:15, ... or another eligible 15-minute interval**. It must choose **exactly one** start, and the job then runs **without interruption** for the assumed duration.

For each possible start we first calculate the *incremental* grid electricity needed to add the job **on top of a separately assumed background load**:

\[
\Delta G_t = \max(0, B_t + J_t - PV_t) - \max(0, B_t - PV_t)
\]

Here, `B` is separately specified other load, `J` is the **assumed** job demand, and `PV` is available solar power. Electricity cost is `Δ grid kWh × €/MWh ÷ 1000`. We **do not** count the TEC forecast twice. The optional carbon preference is a *user-chosen shadow cost*, not a measured hourly CO₂ price.

**MILP:** choose one feasible binary start minimizing incremental wholesale job cost plus the stated preference for lower grid energy. We compare its choice with a direct enumeration of eligible starts to check the solver.

In [7]:
def evaluate_job_starts(slots, cfg, shared_pv=True, shadow_eur_kg=None):
    """Enumerate exact energy balances of a SINGLE non-preemptible hypothetical job."""
    N, L = len(slots), int(cfg["job_slots"])
    if not (1 <= L <= N):
        raise ValueError("Job does not fit in the window")
    job_kw = float(cfg["job_kw"])
    base_kw = float(cfg["other_load_kw"])
    pv = slots.pv_input_kw.to_numpy(dtype=float) if shared_pv else np.zeros(N)
    prices = slots.price_eur_mwh.to_numpy(dtype=float)
    assert np.isfinite(pv).all() and np.isfinite(prices).all()
    if (pv < 0).any():
        raise ValueError("PV input must be nonnegative and available")
    if shadow_eur_kg is None:
        shadow_eur_kg = cfg["carbon_shadow_eur_per_kg"]
    factor = float(cfg["annual_grid_factor_kg_per_kwh"])
    grid_before = np.maximum(base_kw - pv, 0)
    candidates = []
    for start in range(N - L + 1):
        occupied = np.zeros(N, dtype=float)
        occupied[start:start+L] = 1.0
        site_demand_kw = base_kw + job_kw * occupied
        # Constraint evaluated on every occupied slot. No assumed PV is used to
        # conceal a violation of the gross load / feeder-headroom proxy.
        cap_ok = bool((site_demand_kw <= cfg["grid_cap_kw"] + 1e-9).all())
        clock = slots.WsDateTime.iloc[start].strftime("%H:%M")
        manual_ok = clock not in cfg["blocked_start_times"]
        after = np.maximum(site_demand_kw - pv, 0)
        delta_import_kwh = np.maximum(after - grid_before, 0) * SLOT_HOURS
        job_energy_kwh = job_kw * L * SLOT_HOURS
        grid_kwh = float(delta_import_kwh.sum())
        pv_used = float(job_energy_kwh - grid_kwh) if shared_pv else 0.0
        wholesale_eur = float(np.dot(delta_import_kwh, prices) / 1000)
        carbon_kg = grid_kwh * factor
        shadow_cost = float(shadow_eur_kg) * carbon_kg
        candidates.append({
            "Start_index":start,
            "Start":slots.WsDateTime.iloc[start],
            "End":slots.WsDateTime.iloc[start+L-1] + pd.Timedelta(minutes=SLOT_MINUTES),
            "Clock":clock,
            "Feasible":bool(cap_ok and manual_ok),
            "Job_kWh":job_energy_kwh,
            "Grid_kWh":grid_kwh,
            "PV_used_kWh":pv_used,
            "Wholesale_cost_EUR":wholesale_eur,
            "Static_carbon_proxy_kg":carbon_kg,
            "Objective_EUR":wholesale_eur + shadow_cost,
        })
    return pd.DataFrame(candidates)

def solve_job_milp(candidates):
    """Binary one-job MILP. Ineligible starts are fixed at zero by variable bounds."""
    feasible = candidates.Feasible.to_numpy(dtype=bool)
    if not feasible.any():
        return None, "No feasible start. Change approved constraints, not the measurements."
    cost = candidates.Objective_EUR.to_numpy(dtype=float)
    bounds = Bounds(lb=np.zeros(len(cost)), ub=feasible.astype(float))
    exactly_one = LinearConstraint(np.ones((1,len(cost))), lb=[1], ub=[1])
    result = milp(c=cost, integrality=np.ones(len(cost),dtype=int),
                  bounds=bounds, constraints=exactly_one,
                  options={"time_limit":20.0, "mip_rel_gap":1e-9})
    if not result.success or result.x is None:
        return None, f"Solver did not certify an optimum: {result.message}"
    picked = int(np.argmax(result.x))
    if result.x[picked] < 0.99 or not feasible[picked]:
        raise ArithmeticError("MILP binary decision or feasibility check failed")
    direct_min = float(np.min(cost[feasible]))
    if not np.isclose(cost[picked], direct_min, rtol=1e-7, atol=1e-8):
        raise AssertionError("MILP does not match the directly enumerated optimum")
    return picked, str(result.message)

candidates = evaluate_job_starts(window, SCENARIO,
    shared_pv=SCENARIO["assume_shared_pv_bus"])
best_idx, solver_status = solve_job_milp(candidates)
if best_idx is None:
    raise RuntimeError(solver_status)

chosen = candidates.iloc[best_idx]
first_feasible = candidates.loc[candidates.Feasible].iloc[0]
price_only_candidates = evaluate_job_starts(window, SCENARIO,
    shared_pv=SCENARIO["assume_shared_pv_bus"], shadow_eur_kg=0)
price_only_idx, _ = solve_job_milp(price_only_candidates)
price_only = price_only_candidates.iloc[price_only_idx]

savings_eur = float(first_feasible.Wholesale_cost_EUR - chosen.Wholesale_cost_EUR)
import_reduction = float(first_feasible.Grid_kWh - chosen.Grid_kWh)
energy_kwh = float(chosen.Job_kWh)
show_cards([
    ("MILP start · what-if", chosen.Start.strftime("%H:%M"), f"Job ends {chosen.End:%H:%M}; requires operator approval"),
    ("Earliest eligible start", first_feasible.Start.strftime("%H:%M"), "Same job, same fixed energy"),
    ("Grid energy for selected job", f"{chosen.Grid_kWh:.3f} kWh", f"{chosen.PV_used_kWh:.3f} kWh of assumed PV offset"),
    ("Estimated wholesale job cost", f"€{chosen.Wholesale_cost_EUR:.4f}", price_mode),
    ("Cost difference vs earliest", f"€{savings_eur:+.4f}", "Positive means savings; can be negative if carbon preference dominates"),
    ("Solver", "Optimal" if best_idx is not None else "Unavailable", f"{len(candidates):,} candidate starts; {candidates.Feasible.sum():,} feasible"),
])
show_table(candidates.loc[candidates.Feasible].sort_values("Objective_EUR").head(10),
           ["Start", "End", "Job_kWh", "Grid_kWh", "PV_used_kWh", "Wholesale_cost_EUR", "Static_carbon_proxy_kg", "Objective_EUR"],
           caption="Top feasible non-interruptible job starts, ordered by stated objective")
report(f"The model chose **{chosen.Start:%H:%M}–{chosen.End:%H:%M}** instead of the earliest feasible "
       f"**{first_feasible.Start:%H:%M}**. Both complete **{energy_kwh:.3f} kWh** of assumed work. "
       f"The **€{savings_eur:+.4f}** comparison is for one hypothetical job on a "
       f"**selected historical day**, using **{price_mode}**—not a measured saving, a customer bill, or annual ROI.")

**Top feasible non-interruptible job starts, ordered by stated objective**

Start,End,Job_kWh,Grid_kWh,PV_used_kWh,Wholesale_cost_EUR,Static_carbon_proxy_kg,Objective_EUR
25 Mar 2024 08:45,25 Mar 2024 09:45,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 08:30,25 Mar 2024 09:30,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 11:45,25 Mar 2024 12:45,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 11:30,25 Mar 2024 12:30,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 11:00,25 Mar 2024 12:00,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 11:15,25 Mar 2024 12:15,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 10:30,25 Mar 2024 11:30,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 10:45,25 Mar 2024 11:45,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 13:45,25 Mar 2024 14:45,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000
25 Mar 2024 13:30,25 Mar 2024 14:30,1.2000,0.0000,1.2000,0.0000,0.0000,0.0000


> **💡 What this tells us**
>
> The model chose **15:15–16:15** instead of the earliest feasible **08:00**. Both complete **1.200 kWh** of assumed work. The **€+0.0086** comparison is for one hypothetical job on a **selected historical day**, using **actual archived SMARD day-ahead wholesale prices**—not a measured saving, a customer bill, or annual ROI.

### ❓ 8. Why did the optimizer choose that hour?

See how the price of starting the *same job* changes over the candidate hours. When there is enough PV power available behind the assumed shared connection, less electricity must be bought from the grid. The cheapest wholesale start and the lowest-grid-energy start need not be identical when the **cost–carbon preference** is nonzero.

In [8]:
shown = candidates.loc[candidates.Feasible].copy()
fig = go.Figure()
fig.add_trace(go.Bar(x=shown.Start, y=shown.Wholesale_cost_EUR, name="Incremental wholesale cost",
    marker_color=[GREEN if idx == best_idx else BLUE for idx in shown.index],
    hovertemplate="Start: %{x|%H:%M}<br>Cost: €%{y:.4f}<extra></extra>"))
fig.add_trace(go.Scatter(x=[chosen.Start], y=[chosen.Wholesale_cost_EUR], name="MILP choice",
    mode="markers", marker=dict(color=GREEN, size=15, symbol="star")))
fig.add_trace(go.Scatter(x=[first_feasible.Start], y=[first_feasible.Wholesale_cost_EUR],
    name="Earliest start", mode="markers", marker=dict(color=ORANGE,size=12,symbol="diamond")))
fig.update_xaxes(title="Feasible job start · Berlin wall clock")
fig.update_yaxes(title="Added job's wholesale electricity cost (€)")
polish(fig, "What does each feasible start cost?",
       "Bars show the actual price component; green star shows the weighted-objective MILP winner", height=440)

fig = go.Figure()
fig.add_trace(go.Scatter(x=shown.Grid_kWh, y=shown.Wholesale_cost_EUR,
    mode="markers", name="Feasible starts",
    marker=dict(color=PURPLE,size=10,opacity=.72),
    text=shown.Clock, hovertemplate="Start %{text}<br>Grid: %{x:.3f} kWh<br>Cost: €%{y:.4f}<extra></extra>"))
fig.add_trace(go.Scatter(x=[chosen.Grid_kWh], y=[chosen.Wholesale_cost_EUR],
    mode="markers", name="Selected start", marker=dict(color=GREEN,size=16,symbol="star")))
fig.add_trace(go.Scatter(x=[first_feasible.Grid_kWh], y=[first_feasible.Wholesale_cost_EUR],
    mode="markers", name="Earliest start", marker=dict(color=ORANGE,size=13,symbol="diamond")))
fig.update_xaxes(title="Grid energy needed for the job (kWh)")
fig.update_yaxes(title="Incremental wholesale job cost (€)")
polish(fig, "The trade-off: buying less grid power versus paying less",
       "Each dot is the SAME hypothetical job at a different permissible start", height=430)

report("A start with more solar cover can reduce grid imports. A start with cheaper grid prices can "
       "lower the bill even if imports are greater. The cost-only winner is "
       f"**{price_only.Start:%H:%M}**, while the model with the declared environmental preference "
       f"chooses **{chosen.Start:%H:%M}**. We do not convert this static carbon proxy into "
       "a claim of avoided hourly emissions.")

> **💡 What this tells us**
>
> A start with more solar cover can reduce grid imports. A start with cheaper grid prices can lower the bill even if imports are greater. The cost-only winner is **15:15**, while the model with the declared environmental preference chooses **15:15**. We do not convert this static carbon proxy into a claim of avoided hourly emissions.

### ❓ 9. What changes if the PV is not connected to this machine—or the priority changes?

This is essential: **solar power cannot be subtracted from the machine's grid imports without a shared electrical boundary and a verified export/self-consumption rule**. We therefore re-run the same MILP with and without PV sharing. We also compare *minimum wholesale cost* with a clearly labeled *cost + environmental-preference* objective. Missing factory tariff and export arrangements remain outside this model.

In [9]:
variant_specs = [
    ("Shared PV · cost only", True, 0.0),
    ("Shared PV · cost + grid preference", True, SCENARIO["carbon_shadow_eur_per_kg"]),
    ("No PV sharing · cost only", False, 0.0),
    ("No PV sharing · cost + grid preference", False, SCENARIO["carbon_shadow_eur_per_kg"]),
]
variant_rows = []
for label, shared, shadow in variant_specs:
    variants = evaluate_job_starts(window, SCENARIO, shared_pv=shared, shadow_eur_kg=shadow)
    index, message = solve_job_milp(variants)
    if index is not None:
        picked = variants.iloc[index]
        variant_rows.append({"Scenario":label, "Start":picked.Start,
            "Grid_kWh":picked.Grid_kWh, "PV_used_kWh":picked.PV_used_kWh,
            "Wholesale_cost_EUR":picked.Wholesale_cost_EUR,
            "Static_carbon_proxy_kg":picked.Static_carbon_proxy_kg,
            "Objective_EUR":picked.Objective_EUR})
variant_table = pd.DataFrame(variant_rows)
show_table(variant_table, caption="Sensitivity: electrical topology and the declared optimization priority")
report("The no-sharing rows show what happens if IPE_PV **cannot** offset TEC_48S at the site boundary. "
       "Changes in selected start times are properties of the **assumptions**, not proof of an achievable "
       "factory saving. Carbon uses a constant illustrative kg/kWh factor, so only *grid import quantity* "
       "changes here—not the carbon intensity of a particular hour.")

**Sensitivity: electrical topology and the declared optimization priority**

Scenario,Start,Grid_kWh,PV_used_kWh,Wholesale_cost_EUR,Static_carbon_proxy_kg,Objective_EUR
Shared PV · cost only,25 Mar 2024 15:15,0.0000,1.2000,0.0000,0.0000,0.0000
Shared PV · cost + grid preference,25 Mar 2024 15:15,0.0000,1.2000,0.0000,0.0000,0.0000
No PV sharing · cost only,25 Mar 2024 12:00,1.2000,0.0000,0.0742,0.4356,0.0742
No PV sharing · cost + grid preference,25 Mar 2024 12:00,1.2000,0.0000,0.0742,0.4356,0.1265


> **💡 What this tells us**
>
> The no-sharing rows show what happens if IPE_PV **cannot** offset TEC_48S at the site boundary. Changes in selected start times are properties of the **assumptions**, not proof of an achievable factory saving. Carbon uses a constant illustrative kg/kWh factor, so only *grid import quantity* changes here—not the carbon intensity of a particular hour.

### ❓ 10. Does the solution respect its constraints, and what would we save?

Every selected interval is checked again *after* solving. A job cannot be split between times; it must fit the daytime window, satisfy the assumed maximum gross demand, and use the same fixed energy as the baseline. We show energy in **kWh**, power in **kW**, and wholesale cost in **€**.

In [10]:
def job_profile(slots, start_index, cfg, shared_pv):
    active = np.zeros(len(slots), dtype=bool)
    active[int(start_index):int(start_index)+int(cfg["job_slots"])] = True
    demand = np.where(active, cfg["job_kw"], 0.0)
    supply = slots.pv_input_kw.to_numpy(dtype=float) if shared_pv else np.zeros(len(slots))
    base = float(cfg["other_load_kw"])
    extra_grid_kw = np.maximum(base + demand - supply, 0) - np.maximum(base - supply, 0)
    extra_grid_kw = np.maximum(0, extra_grid_kw)
    result = slots[["WsDateTime", "pv_input_kw", "price_eur_mwh"]].copy()
    result["Job_active"] = active
    result["Job_kw"] = demand
    result["Job_grid_kw"] = extra_grid_kw
    result["Job_solar_kw"] = demand - extra_grid_kw
    result["Job_grid_kWh"] = extra_grid_kw * SLOT_HOURS
    result["Job_solar_kWh"] = result.Job_solar_kw * SLOT_HOURS
    return result

optimized_profile = job_profile(window, best_idx, SCENARIO, SCENARIO["assume_shared_pv_bus"])
earliest_profile = job_profile(window, int(first_feasible.Start_index), SCENARIO,
                               SCENARIO["assume_shared_pv_bus"])

# Post-solve verification: physics (under assumptions), binary schedule, exactly one job.
assert int(optimized_profile.Job_active.sum()) == int(SCENARIO["job_slots"])
assert optimized_profile.Job_active.to_numpy().nonzero()[0].tolist() == list(
    range(best_idx, best_idx + SCENARIO["job_slots"]))
assert (SCENARIO["other_load_kw"] + optimized_profile.Job_kw <= SCENARIO["grid_cap_kw"] + 1e-9).all()
assert np.isclose(optimized_profile.Job_grid_kWh.sum() + optimized_profile.Job_solar_kWh.sum(),
                  chosen.Job_kWh, atol=1e-8)
assert np.isclose(optimized_profile.Job_grid_kWh.sum(), chosen.Grid_kWh, atol=1e-8)
assert np.isclose((optimized_profile.Job_grid_kWh * optimized_profile.price_eur_mwh / 1000).sum(),
                  chosen.Wholesale_cost_EUR, atol=1e-8)

chart = go.Figure()
chart.add_trace(go.Bar(x=optimized_profile.WsDateTime, y=optimized_profile.Job_solar_kw,
    name="Hypothetical job supplied by PV", marker_color=GREEN))
chart.add_trace(go.Bar(x=optimized_profile.WsDateTime, y=optimized_profile.Job_grid_kw,
    name="Hypothetical job supplied by grid", marker_color=BLUE))
chart.update_layout(barmode="stack", bargap=0.1)
chart.update_xaxes(title="Historical 15-minute interval · Berlin wall clock")
chart.update_yaxes(title="Job power (kW)", rangemode="tozero")
polish(chart, "Where would the selected job's electricity come from?",
       "Stacked grid and PV power · shared-bus assumption · zero means job is not scheduled", height=450)

change_table = pd.DataFrame([
    {"Case":"Earliest feasible", "Start":first_feasible.Start, "Job_kWh":first_feasible.Job_kWh,
     "Grid_kWh":first_feasible.Grid_kWh, "PV_used_kWh":first_feasible.PV_used_kWh,
     "Wholesale_cost_EUR":first_feasible.Wholesale_cost_EUR,
     "Static_carbon_proxy_kg":first_feasible.Static_carbon_proxy_kg},
    {"Case":"MILP selected", "Start":chosen.Start, "Job_kWh":chosen.Job_kWh,
     "Grid_kWh":chosen.Grid_kWh, "PV_used_kWh":chosen.PV_used_kWh,
     "Wholesale_cost_EUR":chosen.Wholesale_cost_EUR,
     "Static_carbon_proxy_kg":chosen.Static_carbon_proxy_kg},
])
show_table(change_table, caption="Same hypothetical job: earliest versus optimized")
show_cards([
    ("Job energy preserved", f"{energy_kwh:.3f} kWh", "Both schedules complete the same job"),
    ("Wholesale cost difference", f"€{savings_eur:+.4f}", "Hypothetical one-job difference; not realized savings"),
    ("Grid energy difference", f"{import_reduction:+.3f} kWh", "Positive = fewer kWh drawn from grid"),
    ("Static carbon-proxy difference", f"{first_feasible.Static_carbon_proxy_kg - chosen.Static_carbon_proxy_kg:+.3f} kg", "Uses same illustrative factor all day"),
    ("Verified site solar sharing", "Not verified", "Behind-the-meter connection must be checked"),
    ("Machine dispatch authorization", "Not granted", "Must satisfy human-approved operating rules"),
])
report("The mathematical energy balance and one-hour continuity checks passed. "
       "The displayed differences use a **single hand-picked scenario date** and assume fixed job load, "
       "grid headroom and (in the main case) electrical sharing. They are **not measured factory "
       "savings**. Missing electricity prices or PV measurements are never replaced by zero.")

**Same hypothetical job: earliest versus optimized**

Case,Start,Job_kWh,Grid_kWh,PV_used_kWh,Wholesale_cost_EUR,Static_carbon_proxy_kg
Earliest feasible,25 Mar 2024 08:00,1.2000,0.0866,1.1134,0.0086,0.0314
MILP selected,25 Mar 2024 15:15,1.2000,0.0000,1.2000,0.0000,0.0000


> **💡 What this tells us**
>
> The mathematical energy balance and one-hour continuity checks passed. The displayed differences use a **single hand-picked scenario date** and assume fixed job load, grid headroom and (in the main case) electrical sharing. They are **not measured factory savings**. Missing electricity prices or PV measurements are never replaced by zero.

### ❓ 11. What information should be exported, and what still blocks real scheduling?

The output is a **decision-support record**, not a command to equipment. We export every eligible start, a selected-slot profile, and a JSON file recording the chosen assumptions and data sources. These notebook outputs go to `reports/notebook/ml_short_term/notebook_runs`. This makes the example reproducible and gives the future application a clear boundary between validated inputs and hypothetical values.

In [11]:
readiness = pd.DataFrame([
    {"Requirement":"15-minute historical TEC forecast output", "Evidence":"Notebook 01 CSV", "Ready_for_live":"No: one-step historical tests only"},
    {"Requirement":"15-minute historical PV forecast output", "Evidence":pv_mode, "Ready_for_live":"No: multi-hour single-origin PV forecasts missing"},
    {"Requirement":"Time-aware anomaly advisory", "Evidence":advisory_note, "Ready_for_live":"Partial: must check latency, operator signoff and fresh data"},
    {"Requirement":"Electricity prices", "Evidence":price_mode, "Ready_for_live":"Must verify time-of-publication and actual customer tariff"},
    {"Requirement":"Shared electrical meter / PV export contract", "Evidence":"Assumed only", "Ready_for_live":"Not verified"},
    {"Requirement":"Machine job kW, duration, deadlines and setup", "Evidence":"Illustrative SCENARIO", "Ready_for_live":"Not verified"},
    {"Requirement":"Equipment availability and maintenance authorization", "Evidence":"No factory operations source", "Ready_for_live":"Not verified"},
    {"Requirement":"Actual/variable carbon intensity", "Evidence":"Static illustrative factor", "Ready_for_live":"Not available"},
])
show_table(readiness, caption="What is ready as research—and what a factory must provide")

export_candidates = candidates.copy()
export_candidates["Scenario_mode"] = "RETROSPECTIVE_WHAT_IF_NOT_A_LIVE_SCHEDULE"
export_candidates["PV_source"] = pv_mode
export_candidates["Price_source"] = price_mode
export_candidates["Anomaly_context_at"] = decision_time
export_candidates.to_csv(OUT / "milp_candidate_starts_15min.csv", index=False)

export_schedule = optimized_profile.copy()
export_schedule["Selected_start"] = chosen.Start
export_schedule["Mode"] = "RETROSPECTIVE_WHAT_IF_NOT_DISPATCH"
export_schedule["Human_approval"] = bool(SCENARIO["operator_approved"])
export_schedule.to_csv(OUT / "milp_what_if_schedule_15min.csv", index=False)

metadata = {
    "scenario_mode":"RETROSPECTIVE_WHAT_IF_NOT_A_LIVE_SCHEDULE",
    "scenario_date":str(selected_date.date()),
    "decision_time":str(decision_time),
    "chosen_start":str(chosen.Start),
    "first_feasible_start":str(first_feasible.Start),
    "pv_input":pv_mode, "price_input":price_mode,
    "machine_forecast_rows_same_window":int(window.machine_forecast_kw.notna().sum()),
    "overlap_tec_pv_predictions_all_dates":int(paired_count),
    "anomaly_context_known_at_decision":condition_table.assign(
        Latest_available=condition_table.Latest_available.astype(str)).to_dict("records"),
    "verified_savings":False,
    "operator_approval":bool(SCENARIO["operator_approved"]),
    "caveats":["No validated multi-hour forecast from common origin",
       "No verified production schedule, job power or electrical sharing",
       "2024 SMARD day-ahead prices are hourly; missing archival publication times",
       "Static factor is not hourly marginal CO2 intensity"],
    "assumptions":SCENARIO,
}
with (OUT / "milp_what_if_assumptions.json").open("w", encoding="utf-8") as fp:
    json.dump(metadata, fp, indent=2, ensure_ascii=False, default=str)
show_table(pd.DataFrame([
    {"Saved output":"milp_candidate_starts_15min.csv", "Meaning":"All hypothetical starts and exact modeled costs"},
    {"Saved output":"milp_what_if_schedule_15min.csv", "Meaning":"Chosen hypothetical job's interval-by-interval grid/PV split"},
    {"Saved output":"milp_what_if_assumptions.json", "Meaning":"Input sources, human-approval status and missing prerequisites"},
]), caption="Outputs written to reports/ml_short_term/")
report("All exports are clearly marked **RETROSPECTIVE WHAT-IF / NOT DISPATCH**. "
       "A dashboard may show a recommended *candidate*, but the notebook must not "
       "activate a machine or claim guaranteed energy savings. The missing readiness items "
       "need real factory evidence before operational use.")

**What is ready as research—and what a factory must provide**

Requirement,Evidence,Ready_for_live
15-minute historical TEC forecast output,Notebook 01 CSV,No: one-step historical tests only
15-minute historical PV forecast output,historical rolling PV test predictions (different forecast origins),No: multi-hour single-origin PV forecasts missing
Time-aware anomaly advisory,"70,272 timestamped historical condition records","Partial: must check latency, operator signoff and fresh data"
Electricity prices,actual archived SMARD day-ahead wholesale prices,Must verify time-of-publication and actual customer tariff
Shared electrical meter / PV export contract,Assumed only,Not verified
"Machine job kW, duration, deadlines and setup",Illustrative SCENARIO,Not verified
Equipment availability and maintenance authorization,No factory operations source,Not verified
Actual/variable carbon intensity,Static illustrative factor,Not available


**Outputs written to reports/ml_short_term/**

Saved output,Meaning
milp_candidate_starts_15min.csv,All hypothetical starts and exact modeled costs
milp_what_if_schedule_15min.csv,Chosen hypothetical job's interval-by-interval grid/PV split
milp_what_if_assumptions.json,"Input sources, human-approval status and missing prerequisites"


> **💡 What this tells us**
>
> All exports are clearly marked **RETROSPECTIVE WHAT-IF / NOT DISPATCH**. A dashboard may show a recommended *candidate*, but the notebook must not activate a machine or claim guaranteed energy savings. The missing readiness items need real factory evidence before operational use.

### 🏁 12. Final answers — have we learned when the machine should run?

**We have learned how to compare permitted times, not yet how to command a live machine.** This last section summarizes the example, answers the original questions, and makes its assumptions easy to understand even without optimization experience.

In [ ]:
source_summary = ("historic SMARD wholesale prices" if price_mode.startswith("actual")
                  else "an explicitly illustrative, NOT REAL electricity-price scenario")
report(
    f"**1. When does the hypothetical job run best?** For the selected historical "
    f"**{selected_date:%d %b %Y}** scenario, the MILP chose **{chosen.Start:%H:%M}–{chosen.End:%H:%M}** "
    f"for a fixed **{SCENARIO['job_kw']:.2f} kW**, **{SCENARIO['job_slots']*15}-minute** job. "
    f"The earliest allowed start was **{first_feasible.Start:%H:%M}**.\n\n"
    f"**2. Does solar help?** Under the **unverified shared-PV connection** assumption, "
    f"the selected job uses **{chosen.PV_used_kWh:.3f} kWh of PV** and "
    f"**{chosen.Grid_kWh:.3f} kWh of grid electricity**. The no-sharing sensitivity "
    f"shows why the site's electrical boundary must be confirmed.\n\n"
    f"**3. Is it cheaper?** The chosen job's wholesale cost is **€{chosen.Wholesale_cost_EUR:.4f}**, "
    f"with a difference of **€{savings_eur:+.4f}** versus the earliest start, based on "
    f"{source_summary}. This is **not a verified bill saving**; the declared carbon preference "
    f"can trade some cost for fewer grid kWh.\n\n"
    f"**4. Was Notebook 01 used?** Yes, the study loads the actual TEC and PV exported forecasts. "
    f"For this window the PV input is **{pv_mode}**; TEC forecasts exist for "
    f"**{int(window.machine_forecast_kw.notna().sum())} of {len(window)}** slots. "
    f"A fixed job power is assumed rather than misusing the TEC trend as a production job. "
    f"Full planning needs future multi-slot forecasts from one decision origin.\n\n"
    f"**5. Was Notebook 02 used?** Yes. The latest TEC advisory known at the decision "
    f"was **{condition_rows[0]['Advisory']}**, and the PV advisory was "
    f"**{condition_rows[1]['Advisory']}**. Later alerts were not used. "
    f"Neither an alert nor no alert substitutes for operator approval.\n\n"
    f"**6. Is GreenForge ready to schedule production automatically?** **No.** We still need "
    f"verified machine work orders and power, the shared electrical topology, applicable tariffs, "
    f"production deadlines, safety authorization, processing latency, and a genuine multi-interval "
    f"forecast available **before** scheduling. This notebook delivers a transparent "
    f"**decision-support prototype**, not a factory-control instruction."
).

> **💡 What this tells us**
>
> **1. When does the hypothetical job run best?** For the selected historical **25 Mar 2024** scenario, the MILP chose **15:15–16:15** for a fixed **1.20 kW**, **60-minute** job. The earliest allowed start was **08:00**.
> 
> **2. Does solar help?** Under the **unverified shared-PV connection** assumption, the selected job uses **1.200 kWh of PV** and **0.000 kWh of grid electricity**. The no-sharing sensitivity shows why the site's electrical boundary must be confirmed.
> 
> **3. Is it cheaper?** The chosen job's wholesale cost is **€0.0000**, with a difference of **€+0.0086** versus the earliest start, based on historic SMARD wholesale prices. This is **not a verified bill saving**; the declared carbon preference can trade some cost for fewer grid kWh.
> 
> **4. Was Notebook 01 used?** Yes, the study loads the actual TEC and PV exported forecasts. For this window the PV input is **historical rolling PV test predictions (different forecast origins)**; TEC forecasts exist for **0 of 40** slots. A fixed job power is assumed rather than misusing the TEC trend as a production job. Full planning needs future multi-slot forecasts from one decision origin.
> 
> **5. Was Notebook 02 used?** Yes. The latest TEC advisory known at the decision was **not_assessed**, and the PV advisory was **review_observation**. Later alerts were not used. Neither an alert nor no alert substitutes for operator approval.
> 
> **6. Is GreenForge ready to schedule production automatically?** **No.** We still need verified machine work orders and power, the shared electrical topology, applicable tariffs, production deadlines, safety authorization, processing latency, and a genuine multi-interval forecast available **before** scheduling. This notebook delivers a transparent **decision-support prototype**, not a factory-control instruction.

---

### 📚 Research and engineering references

The approach follows industrial energy-flexibility research: compare **the same production task** at different times, account for on-site solar and the price of residual grid demand, and preserve feasibility constraints before claiming benefits.

- **IFAC-PapersOnLine (2020):** *Production scheduling considering dynamic electricity price in energy-efficient factories*. A related factory scheduling study uses **15-minute MILP**, solar generation, dynamic prices and process constraints. [DOI: 10.1016/j.ifacol.2020.12.1823](https://doi.org/10.1016/j.ifacol.2020.12.1823).
- **Energy (2026):** *A forecasting-informed rolling demand response strategy for industrial microgrids*. Highlights why production constraints, **forecasts from a common planning origin**, and rolling updates matter. [DOI: 10.1016/j.energy.2026.141871](https://doi.org/10.1016/j.energy.2026.141871).
- **SciPy:** [Official `scipy.optimize.milp` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.milp.html) for binary decisions, bounds and linear constraints.
- **Bundesnetzagentur / SMARD:** [15-minute wholesale prices available (October 2025)](https://www.smard.de/en/15-minute-wholesale-prices-available-218078). **2024** day-ahead products were **hourly**, even when energy data is shown on a 15-minute grid. [2024 market summary](https://www.smard.de/page/en/topic-article/213922/215704/evaluation-of-last-year).

**Sustainability rule:** More behind-the-meter PV self-consumption can reduce grid imports. A static annual emissions factor can illustrate imported-energy impact, **but cannot prove which quarter-hour has the lowest marginal grid emissions**. No invented tariffs, safety clearance, or future-known telemetry are treated as observed facts.